In [1]:
import numpy as np
from scipy.stats import entropy

def extract_features(pred, image_shape, score_threshold=0.5):

    h, w = image_shape
    image_area = h * w

    boxes = pred["boxes"].detach().cpu().numpy()
    labels = pred["labels"].detach().cpu().numpy()
    scores = pred["scores"].detach().cpu().numpy()

    valid = scores > score_threshold
    boxes = boxes[valid]
    labels = labels[valid]
    scores = scores[valid]

    num_detections = len(boxes)

    box_area_ratios = []
    class_counts = [0,0,0,0]

    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box
        area = (x2-x1)*(y2-y1)
        ratio = area / image_area
        box_area_ratios.append(ratio)

        if label <= 4:
            class_counts[label-1] += 1

    total_damage_area_ratio = sum(box_area_ratios)

    avg_box_area_ratio = np.mean(box_area_ratios) if num_detections > 0 else 0
    max_box_area_ratio = np.max(box_area_ratios) if num_detections > 0 else 0
    min_box_area_ratio = np.min(box_area_ratios) if num_detections > 0 else 0
    var_box_area_ratio = np.var(box_area_ratios) if num_detections > 0 else 0

    damage_density = total_damage_area_ratio / num_detections if num_detections > 0 else 0

    class_prob = np.array(class_counts) / num_detections if num_detections > 0 else np.zeros(4)
    class_entropy = entropy(class_prob) if num_detections > 0 else 0

    mean_confidence = np.mean(scores) if len(scores)>0 else 0

    return [
        num_detections,
        total_damage_area_ratio,
        avg_box_area_ratio,
        max_box_area_ratio,
        min_box_area_ratio,
        var_box_area_ratio,
        damage_density,
        class_counts[0],
        class_counts[1],
        class_counts[2],
        class_counts[3],
        class_entropy,
        mean_confidence
    ]